# Extract some kiwi calls from wild field recordings
---

### Create and activate conda environment

```
conda create -n avianz python==3.12 -c conda-forge -y
conda activate avianz
```

### Run detector over test_audio

In [2]:
# avianz/notebooks/avianz_kiwi_detector.ipynb

import os, glob
import sys
import logging
from pathlib import Path
import subprocess


from avianz.src.core.batch_processor import BatchProcessor, BatchProcessorCallbacks


# === === === C O N F I G === === ===
VERBOSE = True   # Toggle for debug-level logging; set False to suppress debug prints without removing them

# Setup logging in case of errors
logging.basicConfig(
    level=logging.DEBUG if VERBOSE else logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)


# # Setup path configurations
# CONFIG_DIR = r"D:\avianz\avianz\Config"                             # Parent directory of avianz config file
# # CONFIG_FILE = r"D:\avianz\avianz\Config\AviaNZconfig.txt"         # Config file contains needed parameters; can be modified to taste

# AVIANZ_ROOT = Path(r"D:\avianz\avianz")                       # Directory containing main script (AviaNZ.py)
# INPUT_AUDIO_DIR = AVIANZ_ROOT / "test_audio"                  # FALLBACK-TRACK: anchored to AVIANZ_ROOT instead of relative cwd path — cwd resolution was inconsistent across kernel launch contexts
# AUDIO_DIR = str(INPUT_AUDIO_DIR)                                    # Alternate audio source (fewer files for testing purposes) — kept as str for BatchProcessor arg compatibility

# FILTER_DIR = r"D:\avianz\avianz\Filters"                            # Directory containing species-specific wavelet filters (.txt, .h5, .json)

# logging.debug(f"Resolved cwd at runtime: {os.getcwd()}")           # verbose debug print — helps diagnose future relative-path mismatches
# logging.debug(f"INPUT_AUDIO_DIR resolved to: {INPUT_AUDIO_DIR}")


# Setup path configurations
AVIANZ_ROOT = Path("..")                                            # Directory containing main script (AviaNZ.py); relative to avianz/notebooks convention

CONFIG_DIR = AVIANZ_ROOT / "Config"                                 # Parent directory of avianz config file
# CONFIG_FILE = AVIANZ_ROOT / "Config" / "AviaNZconfig.txt"         # Config file contains needed parameters; can be modified to taste

INPUT_AUDIO_DIR = AVIANZ_ROOT / "test_audio"                        # FALLBACK-TRACK: anchored to AVIANZ_ROOT instead of a separately hardcoded relative path
AUDIO_DIR = str(INPUT_AUDIO_DIR)                                    # Alternate audio source (fewer files for testing purposes) — kept as str for BatchProcessor arg compatibility

FILTER_DIR = AVIANZ_ROOT / "Filters"                                # Directory containing species-specific wavelet filters (.txt, .h5, .json)

# FALLBACK-TRACK: checkpoint .pth files were pickled against a bare top-level
# 'src' module path, but current imports resolve through 'avianz.src...'.
# Insert AVIANZ_ROOT into sys.path so bare 'src' becomes importable,
# matching the module path recorded inside the pickled checkpoint.
# This does not modify any source files or checkpoints directly.
if str(AVIANZ_ROOT) not in sys.path:
    sys.path.insert(0, str(AVIANZ_ROOT))
    logging.debug(f"Inserted {AVIANZ_ROOT} into sys.path for bare 'src' module resolution.")

logging.debug(f"Resolved cwd at runtime: {os.getcwd()}")
logging.debug(f"AVIANZ_ROOT resolved to: {AVIANZ_ROOT}")
logging.debug(f"INPUT_AUDIO_DIR resolved to: {INPUT_AUDIO_DIR}")






# Check what files are in the filter directory
files = os.listdir(FILTER_DIR)
# Find available species
species_list = sorted(list(set([os.path.splitext(f)[0] for f in files if '.' in f and not f.startswith('__')])))

print("Available species:")
for s in species_list:
    print(f"- {s}")

# List the calls of our species of interest: North Island Brown Biwi
kiwi_list = [ 
    'Kiwi (Nth Is Brown)',
    'Kiwi (Nth Is Brown)_chp'
    ]


def validate_environment():
    """Validates that folders and necessary files exist before running processes."""
    if not AVIANZ_ROOT.exists():
        raise FileNotFoundError(f"AviaNZ engine directory not found at: {AVIANZ_ROOT}")
    if not (AVIANZ_ROOT / "AviaNZ.py").exists():
        raise FileNotFoundError(f"Could not locate 'AviaNZ.py' source wrapper within {AVIANZ_ROOT}")
    if not INPUT_AUDIO_DIR.exists():
        raise FileNotFoundError(f"Target audio input path does not exist: {INPUT_AUDIO_DIR}")
        
    wav_files = list(INPUT_AUDIO_DIR.glob("*.wav"))
    if not wav_files:
        logging.warning(f"No target .wav files discovered inside {INPUT_AUDIO_DIR}")
    else:
        logging.info(f"Discovered {len(wav_files)} target audio source files for processing.")


# This throws alot of errors so handle them all at once. 
class MockCallbacks:
    def __getattr__(self, name):
        def dummy(*args, **kwargs):
            # Print which UI element the processor is looking for
            print(f"DEBUG: Processor requested UI callback: {name}")
            # Reasonable defaults for common UI interactions
            if "confirm" in name or "ask" in name: return True
            return None
        return dummy

class SilentCallbacks(BatchProcessorCallbacks):
    """Bypasses UI prompts for automated background processing."""
    def ask_resume_analysis(self, message): return True
    def confirm_analysis_launch(self, message): return True
    def update_progress(self, current, total, message): print(f"[{current}/{total}] {message}")



### Build the processor by specifying what we want to search for
# Initialize processor
try: 
    validate_environment()

    processor = BatchProcessor(
        configdir=CONFIG_DIR,       # Set parent didrectory for config file
        directory=AUDIO_DIR,        # Set source directory for audio files
        recognisers=kiwi_list,      # Pass the species list to recogniser
        callbacks=SilentCallbacks(),
    )
    processor.filtersDir = FILTER_DIR  # FALLBACK-TRACK: constructor signature has no filtersDir param
                                        # (confirmed via inspect.signature probe); set as
                                        # post-construction attribute instead

    # # Setup callback
    # processor.callbacks = MockCallbacks() # Bypasses the confirmation GUI

    logging.debug(f"Filter directory: {processor.filtersDir}")

    # # Setup callback
    # processor.callbacks = MockCallbacks() # Bypasses the confirmation GUI

    print(f"DEBUG: Filter directory: {processor.filtersDir}")

    # Check filter dictionary
    processor.FilterDicts = processor.ConfigLoader.filters(processor.filtersDir)
    if processor.FilterDicts is None:
        # FALLBACK-TRACK: previously only printed a debug message and allowed execution
        # to continue into process_files(), which then failed downstream with an
        # unhelpful NoneType subscript error. Now raises immediately with context.
        logging.error(f"ConfigLoader.filters() returned None for filtersDir: {processor.filtersDir}")
        logging.error(f"Requested species/recognisers: {kiwi_list}")
        raise RuntimeError(
            "FilterDicts failed to load. Check that filtersDir matches FILTER_DIR "
            "and that filter files for the requested species parse correctly."
        )
    else:
        logging.info(f"Keys found: {list(processor.FilterDicts.keys())}")


    # Run classifier
    processor.process_files()

    logging.info(f"Classification complete. Annotations .data saved to {INPUT_AUDIO_DIR}")
except Exception as e:
    logging.error(f"Execution failed: {e}", exc_info=True)
    sys.exit(1)

2026-07-06 19:36:36,903 [DEBUG] Resolved cwd at runtime: d:\avianz\avianz\notebooks
2026-07-06 19:36:36,904 [DEBUG] AVIANZ_ROOT resolved to: ..
2026-07-06 19:36:36,905 [DEBUG] INPUT_AUDIO_DIR resolved to: ..\test_audio
Available species:
- Bittern
- Bittern_chp
- Kakapo
- Kiwi (Great Spotted)
- Kiwi (Little Spotted)
- Kiwi (Little Spotted)_21-13-34
- Kiwi (Little Spotted)_chp
- Kiwi (Little Spotted)_syll_M
- Kiwi (Nth Is Brown)
- Kiwi (Nth Is Brown)_chp
- Kiwi (Tokoeka Fiordland)
- LongTailedCuckoo
- Morepork
- Morepork_04-23-42
- Morepork_chp
- NZ Bats
- NZ Bats_NP
- NZBats_NP
2026-07-06 19:36:36,908 [INFO] Discovered 4 target audio source files for processing.
Loading software settings from file ..\Config\AviaNZconfig.txt
Loading call filters from folder ..\Config\./Filters
Folder ..\Config\./Filters not found, no filters loaded
2026-07-06 19:36:36,910 [DEBUG] Filter directory: ..\Filters
DEBUG: Filter directory: ..\Filters
Loading call filters from folder ..\Filters
Loaded filters: 

%tb